# Chapter 34

# Dealing with Messy Time Series 

## Introduction   

In the proceeding 2 chapters we have focused on the key properties of time series data and on creating basic models for time series forecasting. The models we explored decomposed time series into unobserved components, trend, random walk, seasonal and stochastic. This type of model can be effective in many cases, as you have seen in the proceeding chapter. But many real-world time series are show more complex behavior than the basic models allow.  

In this chapter we will explore two important advanced time series model extensions on the basic models:     
1. The models we have used so far, only accommodate decomposition with one seasonal component. However in many cases, there is more than one seasonal period. For these cases, we need a decomposition that will account for multi-period seasonal behavior. Examples of seasonal components which might be simultaneously present in a time series include:     
    - **Hourly variation** is common in many time series, such as modeling demand for business services or temperature series.   
    - **Day of the week variation** is often found in business and economic time series.     
    - **Annual variation** is common in both business and natural time series. Such seasonality is see in both monthly and quarterly data. Business and economic data often show such cycles, which in fact, is the origin of the term seasonal variation. Wildlife populations and other natural time series often have annual components as well.       
2. Up to now, we have only worked with univariate time series models. However, it is very common to have covariates that provide information which add information to the forecasting model. Such variables are often referred to as 'exogounous' since they are not a component of the endogounous time series. The choice of exogounous variables one can use to improve a forecasting model can be quite problem specific. Some examples include:    
    - **Holidays** can have a significant effect on business and economic time series. Holidays can fall on a specific date, such as New Year's Day, A relative calendar date, such as the 4th Thursday in November for the US Thanksgiving holiday. Other holidays can follow a lunar calendar, such as Ramadan or Easter. Further, holiday effects can proceed or lag the actual holiday date. For example, beer and snack purchases rise in the days leading up to the Super Bowl Sunday in the US.      
    - **Weather** can effect both business process, such as ice cream sales, as well as natural processes, such as growth of plants.
3. Time series often have **break points** or **switch points** where the statistical properties of the time series change. For example, time series can have sudden changes in mean and variance at these break points. To accommodate these changes, the break points must be identified and the model changed to match the changed properties of the time series.     
    
In this chapter we will extend the basic time series models to incorporate these effects.     

**Instructions:** Complete the exercises in this notebook. Some exercises require coding. The answers to the exercise questions should be short and succinct. **Most question answers require only one to three sentences**. Use of overly long answers will result in loss of credit. Please thin about the theory behind the question and apply the reasoning required to create an answer. AI use must conform to the course policy as written in the Syllabus. Do not use an LLM or other AI to create your answers to the conceptual questions. 

**Execution of this notebook:** The code cells in this notebook must be run in order. Failure to do so will result in errors.  

**Attribution:** Some of the code in this notebook was modified from code originally developed using Copilot and Gemini, LLMs. 

If you do not have the [sktime package](https://www.sktime.net/) installed, uncomment and execute the code in the cell below.   

In [ ]:
#!pip install -U sktime

To get started, execute the code in the cell below to import required packages and instantiate some utility functions.  

In [ ]:
from math import sin, cos, pi
import pandas as pd
import numpy as np
import numpy.random as nr
from scipy.stats import zscore
import sklearn.linear_model as lm
import statsmodels.tsa.seasonal as sts
from scipy import stats
import statsmodels.tsa.arima_process as arima
from statsmodels.tsa.arima.model import ARIMA, ARIMAResults
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.holtwinters import Holt, ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.deterministic import DeterministicProcess, DeterministicTerm, Fourier, Seasonality, TimeTrend
import statsmodels.api as sm
from scipy.signal import lfilter
from sktime.detection.stray import STRAY
from sktime.detection.bs import BinarySegmentation

from scipy import fft
import calendar
import itertools 
import pmdarima as pm
import statsmodels.graphics.tsaplots as splt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings 

%matplotlib inline

def plot_ts(ts, lab = '', xlab='Date', ax=None, alpha=1.0, linewidth=1.0):
    if ax == None: _,ax = plt.subplots(figsize=(9,4))
    ts.plot(ax=ax, alpha=alpha, linewidth=linewidth)
    ax.set_title('Time series plot of ' + lab)
    ax.set_ylabel('Value')
    ax.set_xlabel(xlab)
    
def dist_ts(ts, lab = '', bins = 40):
    _,ax = plt.subplots(1,2,figsize=(10,4))
    ## Plot the histogram with labels
    ts.hist(ax = ax[0], bins = bins, alpha = 0.5);
    ax[0].set_xlabel('Value');
    ax[0].set_ylabel('Frequency');
    ax[0].set_title('Histogram of ' + lab);
    ## Plot the q-q plot on the other axes
    stats.probplot(ts, plot = ax[1]);
    
def auto_partial_corr_plot(ts):
    _,ax = plt.subplots(2,1, figsize=(8,8))
    _=splt.plot_acf(ts, lags = 40, ax=ax[0]);
    _=splt.plot_pacf(ts, lags = 40, method='yw', ax=ax[1]);
    
def decomp_ts(ts, freq = 'M', model = 'additive'):
    res = sts.seasonal_decompose(ts, model = model) 
    res.plot()
    return(pd.DataFrame({'resid': res.resid, 
                         'trend': res.trend, 
                         'seasonal': res.seasonal},
                       index = ts.index) )        

def test_stationarity(series, regression_model='c', rounding=3):
    """Function to test the stationarity of time series using:
    1. Augmented Dicky Fuller test using AIC 
    2. KPSS test for constant
    
    Arguments:  
        Series: the time series to be evaluated.   
        regression_model: model to be used:   
            'c' - model with a constant   
            'ct' - model wiht a constant and trend  
        rounding: the number of digits to display in printed output.   
    """
    if regression_model not in ['c','ct']:    
        print('Error!: argument regression model must be one of {\'c\',\'ct\'}')
        return
    with warnings.catch_warnings(): 
        warnings.filterwarnings("ignore")
        kpss_test = kpss(series, regression=regression_model)
        ADF_test = adfuller(series, regression=regression_model, autolag="AIC")
    return [round(ADF_test[0],rounding), round(ADF_test[4]['5%'],rounding), round(ADF_test[1],rounding),
                                       round(kpss_test[0],rounding), round(kpss_test[1],rounding)]

def stationary_tests(series, names, regression_model='c', rounding=3):
    test_columns = ['ADF_statistic','ADF_conf_int','ADF_p_value','KPSS_statistic','KPSS_p_value']
    out = pd.DataFrame(index=names, columns=test_columns)    
    for ser, name in zip(series,names):  
        out.loc[name,:] = test_stationarity(ser, regression_model=regression_model, rounding=rounding)
    return out

def auto_partial_corr_plot(ts, lags=40):
    _,ax = plt.subplots(2,1, figsize=(8,8))
    _=splt.plot_acf(ts, lags=lags, ax=ax[0]);
    _=splt.plot_pacf(ts, lags=lags, method='yw', ax=ax[1]);
    

def dist_ts(ts, lab = '', bins = 40):
    _,ax = plt.subplots(1,2,figsize=(10,4))
    ## Plot the histogram with labels
    ts.hist(ax = ax[0], bins = bins, alpha = 0.5);
    ax[0].set_xlabel('Value');
    ax[0].set_ylabel('Frequency');
    ax[0].set_title('Histogram of ' + lab);
    ## Plot the q-q plot on the other axes
    stats.probplot(ts, plot = ax[1]);


## Example Data

We will not generate a first, fairly simple, univariate time series with two seasonal periods, weekly and annual. The series is constructed by summing sinusoidal components. Such multiperiod seasonal behavior is found in many real-world time series.  

Execute the code in the cell below to generate the series and display the result.  

In [ ]:
## Create a synthetic time series with multiple seasonality 
#ts = pd.Series(np.random.rand(8 * 365), index=pd.date_range(start='2018-01-01', periods= 8 * 365, freq='D'))
n = 8*365

# Generate AR(1) component: y_t = phi * y_{t-1} + epsilon_t
phi = 0.5  # Autoregressive parameter (must be between -1 and 1)
noise = np.random.normal(loc=0, scale=0.5, size=n)
ar1 = lfilter([1], [1, -phi], noise)

## Create a synthetic time series with AR(1) behavior
ts = pd.Series(ar1, index=pd.date_range(start='2018-01-01', periods=n, freq='D'))

ts += 2.0 * np.sin(np.arange(len(ts)) * 2 * np.pi / 7) +  \
      0.5 * np.cos(np.arange(len(ts)) * 2 * (np.pi + 0.5) / 7) +  \
      2.0 * np.cos(np.arange(len(ts)) * 2 * np.pi / 7)    # Weekly seasonality
ts += np.sin(np.arange(len(ts)) * 2 * np.pi / 365.25) + \
      0.5 * np.cos(np.arange(len(ts)) * 2 * np.pi / 365.25)  # Annual seasonality   

_,ax = plt.subplots(2, 1, figsize=(10,7))
ts.plot(ax=ax[0])
ax[0].set_title('Time series with multiple seasonality  ');
ax[0].set_ylabel('Value');
ax[0].set_xlabel('Date');
ts[:'2019-01-01'].plot(ax=ax[1])
ax[1].set_title('Segment of time series with multiple seasonality  ');
ax[1].set_ylabel('Value');
ax[1].set_xlabel('Date');

The plots clearly show both the annual and weekly seasonality in the time series. 

## Fourier Components and Power Spectrum of Time Series

A useful way to understand the frequency components of a time series is to examine its power spectrum. This is done by decomposing the time series into its sinusoidial components of different frequencies. This decomposition is know as a [**Fourier transform**](https://en.wikipedia.org/wiki/Fourier_transform). We will not purse any detail in this discussion, going only far enough to gain a basic ability to interpret the results.   

To compute a Fourier transform on a discretely sampled data, such as digital time series data, the [**discrete Fourier transform (DFT)**](https://en.wikipedia.org/wiki/Discrete_Fourier_transform). For a one dimensional sequential sample, $\mathbf{x} = [x_1, x_2, x_3, \ldots, x_n]$, we can decompose this sequence into a sequence of complex frequency components, $\mathbf{\Omega} = [\omega_0, \omega_1, \omega_2, \ldots, \omega_{N-1}]$. The basic DFT for one frequency component, $X_k$, is formulated as:        

$$\Omega_k = \sum_{n=0}^{N-1} x_n \cdot e^{-\frac{i2\pi}{N}kn}$$

How can we understand this complex looking relationship? First, recall that the $i$ symbol indicates a complex number. Next, it is useful to note that we can represent a complex exponential function into sin and cos function using [**Euler's formula**](https://en.wikipedia.org/wiki/Euler%27s_formula):

$$
e^{-i\ q} = cos(q) - i\ sin(q)\\   
$$

Applying Euler's formula to the DFT, we get the following relation:  

$$
\Omega_k = \sum_{n=0}^{N-1} x_n cos \Bigg(\frac{2 \pi}{N}kn \Bigg) - i\ x_n\ sin \Bigg( \frac{2 \pi}{N}kn \Bigg)
$$ 

From the relationships of the DFT we can see that it is a linear superposition of the complex sinusoidaL harmonics with the sequence $\mathbf{x}$. The coefficient $k$ determines the harmonic. The larger the value of $k$ determines the harmonic. Each component, $x_k$, represents the a coefficient for a specific frequency and phase. Another way to understand the DFT is as a convolution between the sinusoids of different frequencies and phases and the sequence, $\mathbf{x}$. Phase is represented by the complex value.  

For our purposes here, we are not interested in the complex representation. Rather, we are interested in the **power spectrum**. The power spectrum is the square of the magnitude of the spectral components. The power spectrum is symmetric about zero. Each frequency component is represented on both the positive side and the negative side.   

### Example of Fourier Decomposition of a Time Series  

The code in the cell below computes and plots the Fourier power spectrum of the example time series. Execute this code and examine the result.  

In [ ]:
def compute_power_spectrum(series, log_power=False): 
    ## Compute the Fourier transform and power spectrum
    edge_fft = np.fft.fft(series)
    if log_power: 
        return np.log(np.square(np.abs(fft.fftshift(edge_fft))))
    else:    
        return np.square(np.abs(fft.fftshift(edge_fft)))

def series_spectrum_plot(series, xlim=(-600, 600), years=8, log_power=False): 
     ## Compute the axis lables for the power specture  
    frequencies = int(len(spectrum)/2)
    frequencies = range(-frequencies, frequencies, 1)
    frequencies = [f/(years/8) for f in frequencies] # Scale the frequencies 

    ## Plot the time series and the power spectrum     
    _,ax = plt.subplots( figsize=(12,3))
    ax.plot(frequencies, spectrum);
    if log_power: 
        ax.set_ylabel('Log Power')
    else: 
        ax.set_ylabel('Power')
    ax.set_xlabel('frequency')
    ax.set_xlim(xlim[0], xlim[1])

## Compute the Fourier transform and power spectrum
spectrum = compute_power_spectrum(ts)

## Plot the time series and the power spectrum     
series_spectrum_plot(spectrum, xlim=(-2*365, 2*365))
series_spectrum_plot(spectrum,  xlim=(-2*7, 2*7))

> **Exercise 34-1:** What evidence do you see of the presence of the seasonal periods in the Fourier decomposition of the time series?     

> **Answer:**      

### Construction of Time Series from Fourier Harmonics    

For real-valued time series we can construct complex waveforms from a series of harmonics. For $N$ harmonics, $k = 1, 2, \dots, N$, a base frequency, $f_0$ , DC or bias term $a_0$, and $2 N$ weights, $a_k$ and $b_k$, the waveform is constructed:     

$$f(t) = a_0 + \sum_{k=1}^{N} a_k\ sin (2 \pi k f_0 t) - b_k\ cos ( 2 \pi k f_0 t )$$

To get a feel for this construction, we will show an example. In this example, we will build a time series waveform with a relatively high and narrow peak using five harmonics. The five harmonics, multiplied by the weights, are plotted. Then the approximations of the waveform with increasing numbers of harmonics are plotted.  

In [ ]:
def generate_harmonic_series(t, f0, a0, coeffs):
  """Computes a time series from harmonic sine and cosine components.

  Args:
      t (np.ndarray): Time vector.
      f0 (float): Fundamental frequency (1 / period).
      a0 (float): Baseline offset (DC component).
      coeffs (list of tuples): [(a1, b1), (a2, b2), ...] for each harmonic k.

  Returns:
      np.ndarray: Computed time series.
  """
  y = np.full_like(t, a0, dtype=float)

  for k, (a_k, b_k) in enumerate(coeffs, start=1):
    omega_k = 2 * np.pi * k * f0
    y += a_k * np.cos(omega_k * t) + b_k * np.sin(omega_k * t)

  return y


# 1. Initialize time vector, fundamental frequency, and baseline offset
t = np.linspace(0, 2, 100)  # 2 seconds (or 2 cycles) sampled at 100 points
f0 = 1.0  # Fundamental frequency (1 cycle per unit time)
a0 = 0.0  # Baseline offset (DC component)

# 2. Define (a_k, b_k) cosine and sine coefficients for harmonics k = 1 to 5
harmonics_5 = [
    (1.0, 0.5),  # k=1 (fundamental frequency f0)
    (0.6, 0.3),  # k=2 (2 * f0)
    (0.4, 0.2),  # k=3 (3 * f0)
    (0.2, 0.1),  # k=4 (4 * f0)
    (0.1, 0.05),  # k=5 (5 * f0)
]

# Compute time series progressively from 1 to 5 harmonics
series_dict = {}
for n in range(1, 6):
  series_dict[n] = generate_harmonic_series(t, f0, a0, harmonics_5[:n])

# Plot the individual harmonics
v_lines = [0.25*x for x in range(9)] 
print(v_lines)
_, ax = plt.subplots(5, 1, figsize=(12, 15), sharex=True)
for n in range(1, 6):
  if n == 1:
    # Subtract the constant offset a0 from the first cumulative series
    component = series_dict[1] - a0
  else:
    # Subtract the previous cumulative sum to isolate harmonic n
    component = series_dict[n] - series_dict[n - 1]

  ax[n - 1].plot(t, component)
  ax[n - 1].set_title(f'Individual Harmonic Component {n}')
  ax[n - 1].set_ylabel('Amplitude')
  ax[n - 1].set_ylim(-1.2,1.2)
  for v_line in v_lines:
      ax[n - 1].axvline(v_line, c='red', ls='--')
ax[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()



# Plot the time series as the harmonics are added 
plt.figure(figsize=(10, 6))
for n, y in series_dict.items():
  plt.plot(t, y, label=f"{n} Harmonic{'s' if n > 1 else ''}")
for v_line in v_lines:
      plt.axvline(v_line, c='red', ls='--')

plt.title("Time Series Progression: 1 to 5 Harmonics")
plt.xlabel("Time (t)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Exercise 34-2:** Describe the effect of adding more harmonics with respect to creating a waveform with a sharp and narrow peak.     

> **Answer:**      

## Apply MSTL

The **multiple seasonal trend decomposition using LOESS (MSTL)** algorithm allows us into the following components.     
1. A nonlinear trend component computed using LOESS local regression.
2. Multiple seasonal components at the periods specified.
3. The residual of the time series with the trend and seasonal components removed.  


The code in the cell below uses the [MSTL implementation in statsmodels.tsa](decomposition using LOESS). 

In [ ]:
window = [3, 3]
periods = [7, 365]
multi_period_decomposion = sts.MSTL(ts, 
                                    periods = periods,
                                    windows = window,
                                   ).fit()


multi_period_decomposion.plot()
plt.tight_layout()
plt.show()

_,ax = plt.subplots(figsize=(10,3))
multi_period_decomposion.seasonal.loc[:'2019-01-01','seasonal_7'].plot(ax=ax)
ax.set_title('One year of weekly seasonal compoent');
ax.set_ylabel('Value');
ax.set_xlabel('Date');

## Time Series With More Complex Seasonal Components

In the foregoing we have generated and explored a time series with simple seasonal behavior with two periods. We will now create and explore a time series with a more complex seasonal components. These components are computed:       

1. The weekly seasonal component is created with [statsmodels.tsa.deterministic.DeterministicProcess](https://www.statsmodels.org/devel/generated/statsmodels.tsa.deterministic.DeterministicProcess.html). This function allows construction of the deterministic seasonal components of a time series by linear superposition of Fourier terms.
2. The annual seasonal component is created from two Fourier harmonics.
3. Autoregressive (AR) components are computed to ensure the time series is causal.
4. The complete series is computed as the sum of these components, or linear superposition. 

Execute the code in the cell below to generate the time series and display the results.    

In [ ]:
series_length = 8 * 365  
date_idx = pd.date_range(start='2018-01-01', periods=series_length, freq='D')

# 2. Define exact 365.25 annual Fourier term (order=2 adds 4 columns: 2 sin, 2 cos)
annual_fourier = Fourier(period=365.25, order=2)

# 3. Combine period=7 and period=365.25 into one DeterministicProcess with a single constant
det_proc = DeterministicProcess(
    date_idx, 
    constant=True, 
    period=7, 
    fourier=2,
    additional_terms=[annual_fourier]
)
det_terms = det_proc.in_sample()

# 4. Generate AR(1) error process to ensure stable ARIMA model: u[t] = 0.9 * u[t-1] + eps[t] 
gen = np.random.default_rng(98765432101234567890)
eps = gen.standard_normal(series_length)
ar_errors = np.zeros(series_length)
ar_errors[0] = eps[0]

for i in range(1, series_length):
    ar_errors[i] = 0.9 * ar_errors[i - 1] + eps[i]

# 5. Combine deterministic seasonal signal + AR(1) errors
# Total columns = 1 (const) + 4 (period=7, fourier=2) + 4 (period=365.25, order=2) = 9
params = np.array([
    1.0,                  # const
    3.0, -1.0, 4.0, -2.0, # weekly seasonality (period=7)
    5.0, -3.0, 2.0, -1.5  # annual seasonality (period=365.25)
])
det_term = det_terms.to_numpy() @ params + ar_errors
ts_det = pd.Series(det_term, index=date_idx, name='synthetic_series')

# 6. Plot full series and first year
fig, ax = plt.subplots(2, 1, figsize=(12, 8))
ts_det.plot(ax=ax[0], title='Full Synthetic Series: Period 7 + 365.25 (2018–2025)')
ts_det.loc[:'2019-01-01'].plot(ax=ax[1], title='First Year Zoom (2018)')
plt.tight_layout()
plt.show()

## Compute the Fourier transform and power spectrum
spectrum = compute_power_spectrum(ts_det)
## Plot the time series and the power spectrum     
series_spectrum_plot(spectrum, xlim=(-4*365, 4*365))
series_spectrum_plot(spectrum,  xlim=(-2*7, 2*7))

To further explore this time series, execute the code in the cell below to compute and display the MSTL of the time series with complex deterministic behavior.   

In [ ]:
window = [3, 3]
periods = [7, 365]
multi_period_decomposion_det = sts.MSTL(ts_det, 
                                    periods = periods,
                                    windows = window,
                                   ).fit()


multi_period_decomposion_det.plot()
plt.tight_layout()
plt.show()

_,ax = plt.subplots(figsize=(10,3))
multi_period_decomposion_det.seasonal.loc[:'2019-01-01','seasonal_7'].plot(ax=ax)
ax.set_title('One year of weekly seasonal compoent');
ax.set_ylabel('Value');
ax.set_xlabel('Date');

> **Exercise 34-3:** Examine these results and answer the following questions.
> 1. What specific evidence can you describe that the time series is not just composed of simple sinusoidal seasonal components?
> 2. Consider that the complex nature of the weekly component leads to the question, why are there are two high frequency components in the Fourier spectrum? Based on the weekly seasonal components explain this behavior. 
> 3. Examine the amplitude of the residual and trend (drift in this case) terms compared to the amplitude of the seasonal components. Explain what this difference tells you about how well the MSTL process has isolated the seasonal components and the possible bias?   

> **Answers:**
> 1.     
> 2.      
> 3.     

## SARIMAX Model For Multiseasonality

The basic SARIMAX model allows for a single seasonal differencing term for a single seasonal period. But we can easily extend SAEIMAX to multiple seasonality time series, using exogonous variables.    

### SARIMAX moodel for simple periodic components.  

For the first time series example the annual periodic behavior is simple sinusoid. Therefore we can use a single pair of [Fourier harmonics](https://en.wikipedia.org/wiki/Harmonic_analysis) to model this behavior. For more complex seasonal patterns we will need additional harmonics.   

The weekly seasonal behavior is modeled by using a difference operator with period of 7.  

The code in the cell below creates and displays an exogonus pair of Fourier harmonics.  

In [ ]:
## Create exogonous variable for annual seasonality 
annual_fourier_terms = pd.DataFrame({
    'sin_annual_1': np.sin(np.arange(len(ts)) * 2 * np.pi / 365.25),
    'cos_annual_1': np.cos(np.arange(len(ts)) * 2 * np.pi / 365.25)
})
annual_fourier_terms.index = ts.index

_,ax = plt.subplots(figsize=(10,3))
for col in ['sin_annual_1', 'cos_annual_1']:
 annual_fourier_terms[col].plot(ax=ax, label=col)
ax.set_title('Fourier components of annual seasonality');
ax.set_ylabel('Value');
ax.set_xlabel('Date');
plt.legend()

The code in the cell below instantiates and fits a SARIMAX model accounting for the twp seasonal periods:    
1. The weekly period (7 days) is modeled by first-order differencing.
2. The annual component is modeled using the exogounous Fourier harmonic variables.

The last year of the time series is withheld as test data.  

In [ ]:
SARIMAX_model = SARIMAX(ts[:'2024-12-31'],
                exog=annual_fourier_terms[:'2024-12-31'], # for annual seasonality 
                order=(1, 0, 0),
                seasonal_order=(0, 1, 0, 7), # for weekly seasonality 
           #     trend='c',
                enforce_stationarity=False,
                enforce_invertibility=False).fit()

SARIMAX_model.summary()

The code in the cell below computes the SARIMAX model prediction of the last year of the time series. Since this last year of the time series was not used for model training, this is an out of sample estimate. Some summary statistics are computed an displayed. The actual time series, the prediction and the prediction confidence intervals are all displayed.   

In [ ]:
def plot_forecast(model, time_series, exog, start_date):

    test_length = len(time_series[start_date:])
    forecast = model.get_forecast(steps=test_length, exog=exog[start_date:])

    CIs = forecast.conf_int()

    _,ax = plt.subplots(2, 1, figsize=(10,12))
    print(ax.shape)
    time_series[start_date:].plot(ax=ax[0], c='red', linestyle='--', label='Actual')
    
    forecast_mean = forecast.predicted_mean
    forecast_mean.plot(ax=ax[0])
    resid_series = forecast_mean - time_series[start_date:]
    print('RMSE: ' + str(np.std(forecast_mean - time_series[start_date:])))
    print(pd.Series(test_stationarity(resid_series, regression_model='c', rounding=3), 
                   index=['ADF_statistic','ADF_conf_int','ADF_p_value','KPSS_statistic','KPSS_p_value']))

    ax[0].fill_between(CIs.index,
             CIs.iloc[:,0],
             CIs.iloc[:,1], 
             color="lightblue", 
             alpha=0.3)
    ax[0].set_title('Time series forecast with confidence intervals');
    ax[0].set_ylabel('Value');
    ax[0].set_xlabel('Date');
    ax[0].legend()

    resid_series.plot(ax=ax[1])
    ax[1].axhline(y=0, c='red', ls='--')
    ax[1].set_title('Residuals vs. time')
    ax[1].set_ylabel('Residual');
    ax[1].set_xlabel('Date');

plot_forecast(SARIMAX_model, ts, annual_fourier_terms, start_date='2025-01-01')

> **Exercise 34-4:** Examine the results of the modeling and answer these questions.
> 1. Are all the model coefficients statistically significant, what does this tell you about the annual periodic behavior and why?       
> 2. Are the out of sample residuals homoscedastic and why?
> 3. Describe the behavior of the prediction confidence intervals during the prediction period.    

> **Answers:**
> 1.    
> 2.     
> 3.      

### SARIMAX model for complex seasonal behavior       

The code in the cell below constructs a SARIMAX model using the seasonal terms for the annual and weekly behavior. No differencing operator is used in this case. An AR(1) component is added to the model. Execute the code and examine the results.    

In [ ]:
train_end = '2024-12-31'
SARIMAX_model_det = SARIMAX(
    ts_det.loc[:train_end],
    exog=det_terms.loc[:train_end],
    order=(1, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit()

SARIMAX_model_det.summary()

Execute the code in the cell below to display model evaluation information.   

In [ ]:
test_exog = det_terms.loc['2025-01-01':]
plot_forecast(SARIMAX_model_det, ts_det, test_exog, start_date='2025-01-01')

> **Exercise 34-5:** Examine the results of the modeling and answer these questions.
> 1. Are all the model coefficients statistically significant, what does this tell you about the annual periodic behavior and why?       
> 2. Are the out of sample residuals homoscedastic and why?
> 3. Describe the behavior of the prediction confidence intervals during the prediction period. State a reason for the observed behavior.  

> **Answers:**
> 1.     
> 2.     
> 3.    

A question one should ask is if adding additional seasonal terms could improve the fit of the deterministic components of the time series. The code in the cell below creates an expanded set of seasonal components which are used as exogenous variables for a SARIMAX model. Execute the code to fit the model.     

In [ ]:
# Define the annual Fourier component (period=365.25 or 365)
annual_fourier = Fourier(period=365.25, order=3)

# Build a eterministicProcess the both weekly period
# The annual terms are included.   
dp = DeterministicProcess(
    index=ts_det.index,
    constant=True,
    period=7,
    fourier=3,
    additional_terms=[annual_fourier]
)

#  Extract the complete exogenous DataFrame 
full_exog = dp.in_sample()
future_exog = dp.out_of_sample(steps=365)


# Fit a much faster standard ARIMAX model (no seasonal_order)
train_end = '2024-12-31'
SARIMAX_model_det = SARIMAX(
    ts_det.loc[:train_end],
    exog=full_exog.loc[:train_end],
    order=(1, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit()

SARIMAX_model_det.summary()

> **Exercise 34-6:** Examine the results and answer these questions.
> 1.  Are all the model coefficients statistically significant, what does this tell you about the fit of the model?
> 2.  Compare the AIC and BIC of the previous model with with fewer seasonal components to the model just fit with more components. What does this comparison tell you about the information in the additional seasonal components?     

> **Answers:**
> 1.     
> 2.    

There is another approach we can take to the seasonal behavior, seasonal differencing. The code in the cell below uses sinusoidal harmonics for the annual seasonality and differencing to account for the weekly seasonality. Execute this code and examine the results.  

In [ ]:
# 1. Build exogenous features using ONLY annual harmonics and NO constant
dp = DeterministicProcess(
    index=ts_det.index,
    constant=False,      # Omit constant because D=1 seasonal differencing removes intercepts
    period=365.25,       # Annual seasonality only
    fourier=2            
)
annual_exog = dp.in_sample()

# 2. Fit SARIMAX with a 7-day seasonal difference (D=1, s=7) and AR(1) for the 0.9 persistence
train_end = '2024-12-31'
sarimax_diff7 = SARIMAX(
    ts_det.loc[:train_end],
    exog=annual_exog.loc[:train_end],
    order=(1, 0, 0),              # Captures the AR(1) persistence
    seasonal_order=(0, 1, 0, 7),  # Applies a 7-day seasonal difference (y_t - y_{t-7})
    enforce_stationarity=False,
    enforce_invertibility=False
).fit()

sarimax_diff7.summary()

Execute the code below to explore the model fit.  

In [ ]:
test_exog = annual_exog.loc['2025-01-01':]
plot_forecast(sarimax_diff7, ts_det, test_exog, start_date='2025-01-01')

## Anomaly Detection    

Messy time series often contain anomalies. There is a vast literature and many implementations of time series anomaly detection algorithms. Without going into any depth we will look at methods to detect two common time series anomalies.    
1. **Outliers**, or unusual values.     
2. **Change points** where a time series experiences a shift in statistical properties such as mean, trend, variance, etc.  

### Outlier Detection

As a first example we will add some outliers to the multiseasonal time series.  

In [ ]:
np.random.seed(58199)
outliers_index = np.sort(np.random.choice(range(len(ts)), size=50, replace=False))
outliers = np.array([0.0]*len(ts))
outliers[outliers_index] = 2.0
ts_outliers = ts + outliers

_,ax = plt.subplots(figsize=(10,3))
ts_outliers.plot(ax=ax)
ax.set_title('Time series with outliers');
ax.set_ylabel('Value');
ax.set_xlabel('Date');

We will not instantiate and fit a [STRAY model from the sktime package](https://www.sktime.net/en/latest/api_reference/auto_generated/sktime.detection.stray.STRAY.html). There are two hyperparameters:   
1. The number of near neighbors, $k$, uses to compute distance gaps.    
2. The confidence, $\alpha$, for the hypothesis test used to test the significance of the outlier.    

The code in the cell below iterates over several hyperparameter values, $k= {2, 3, 5, 9}$ and $\alpha = {0.05, 0.1}$.  

In [ ]:
for k in [2, 3, 5, 9]: 
    _, ax = plt.subplots(1,2, figsize=(12,4))
    for i, alpha in enumerate([0.05, 0.1]):
        X = ts_outliers.to_numpy()
        model = STRAY(alpha = alpha, k=k)
        y = model.fit_transform(ts_outliers)
        detected = ts_outliers[y]

 #       _,ax[i] = plt.subplots(figsize=(10,3))
 #       ts_outliers.plot(ax=ax[i])
 #       ax[i].scatter(detected.index, detected, c='red')
        ts_outliers.plot(ax=ax[i])
        ax[i].scatter(mdates.date2num(detected.index), detected.values, c='red', zorder=5)
        ax[i].set_title('Time series with outliers. with K: ' + str(k) + '  alpha: ' + str(alpha));
        ax[i].set_ylabel('Value');
        ax[i].set_xlabel('Date');
plt.tight_layout()
plt.show()

> **Exercise 34-7:** Examine these results and answer the following questions
> 1. Describe how the anomalies detected changed with the number of neighbors, and what does this tell you about the stability of the detections.
> 2. In this case, does the confidence change the anomalies detected? If there had been a change in the anomalies detected what would this difference means?      

> **Answers:**
> 1.    
> 2.     

### Change Point Detection 

We will now try a simple example of change point detection in a univariate time series.    

The code in the cell below adds a single change point to the multiperiod time series and displays the result.    

In [ ]:
ts_break = ts.copy()
ts_break['2024-01-01':] = ts['2024-01-01':] - 2.0

_,ax = plt.subplots(figsize=(10,3))
ts_break.plot(ax=ax)
ax.set_title('Time series with break point');
ax.set_ylabel('Value');
ax.set_xlabel('Date');

The code in the cell below uses the [BinarySegmentation algorithm from the sktime package](https://www.sktime.net/en/latest/api_reference/auto_generated/sktime.detection.bs.BinarySegmentation.html) to locate the change point in the time series. The result is displayed as a vertical red line. 

In [ ]:
model = BinarySegmentation(threshold=30)
predicted_change_points = model.fit_predict(ts_break)

_,ax = plt.subplots(figsize=(10,3))
ts_break.plot(ax=ax)
ax.axvline(ts_break.index[predicted_change_points.iloc[0].iloc[0]], 
           ymin = np.min(ts_break),
           ymax = np.max(ts_break),
           c = 'red')
ax.set_title('Time series with outliers');
ax.set_ylabel('Value');
ax.set_xlabel('Date');

The change-point has been directed and at least empirically looks reasonable. The question is, is this result statistically significant? The change-point detection algorithm is inherently nonlinear and therefore there is no built in capability to compute confidence intervals. As an alternative we can do the following:   
1. Compute the mean values of the time series along with standard errors before and after the change-point.   
2. Compute a bootstrap confidence interval for the index of the change-point.   

Execute the code in the cell below. Execution may take some time.  

In [ ]:
# Clean conversion to 1D array
y = np.asarray(ts_break).flatten()

if isinstance(predicted_change_points, (pd.Series, pd.DataFrame)):
    cp_indices = predicted_change_points.to_numpy().flatten().astype(int).tolist()
else:
    cp_indices = np.asarray(predicted_change_points).flatten().astype(int).tolist()

# Sort to guarantee order
cp_indices = sorted(cp_indices)

# --- 2. Compute Segment Means & Analytical CIs ---
segments = np.split(y, cp_indices)
fitted_signal = np.zeros_like(y, dtype=float)

print("\n--- Segment Parameter Confidence Intervals (95% CI) ---")
start_idx = 0
for i, seg in enumerate(segments):
    end_idx = start_idx + len(seg)
    mean = np.mean(seg)
    fitted_signal[start_idx:end_idx] = mean
    
    if len(seg) > 1:
        sem = stats.sem(seg)
        ci_low, ci_high = stats.t.interval(0.95, df=len(seg) - 1, loc=mean, scale=sem)
        print(f"Segment {i} [{start_idx}:{end_idx}]: Mean = {mean:.3f}, 95% CI = ({ci_low:.3f}, {ci_high:.3f})")
    else:
        print(f"Segment {i} [{start_idx}:{end_idx}]: Mean = {mean:.3f}, 95% CI = N/A")
    start_idx = end_idx

# --- 3. Robust Bootstrap for Change Point Location CIs ---
if len(cp_indices) > 0:
    residuals = y - fitted_signal
    n_boot = 500
    
    # Initialize a list of lists to track bootstrap locations for each original CP
    boot_history = {cp: [] for cp in cp_indices}
    rng = np.random.default_rng(42)

    for _ in range(n_boot):
        # Generate bootstrap sample
        resampled_y = fitted_signal + rng.choice(residuals, size=len(y), replace=True)
        b_cp = model.fit_predict(pd.Series(resampled_y))
        
        if isinstance(b_cp, (pd.Series, pd.DataFrame)):
            b_idx = b_cp.to_numpy().flatten().astype(int).tolist()
        else:
            b_idx = np.asarray(b_cp).flatten().astype(int).tolist()
            
        # Robust Matching: Assign each detected bootstrap CP to the NEAREST original CP
        for b_point in b_idx:
            closest_orig_cp = min(cp_indices, key=lambda x: abs(x - b_point))
            # Only associate if it's reasonably close (e.g., within 20% of total length) to avoid noise mapping
            if abs(b_point - closest_orig_cp) < (len(y) * 0.2):
                boot_history[closest_orig_cp].append(b_point)

print("\n--- Change Point Location Confidence Intervals (95% Bootstrap CI) ---")
for cp in cp_indices:
        associated_points = boot_history[cp]
        # We need enough successful allocations to compute stable percentiles
        if len(associated_points) > (n_boot * 0.1): 
            low, high = np.percentile(associated_points, [2.5, 97.5])
            print(f"Change Point at Index {cp}: 95% CI = ({low:.1f}, {high:.1f}) [based on {len(associated_points)} samples]")
        else:
            print(f"Change Point at Index {cp}: 95% CI could not be determined reliably (insufficient bootstrap matches).")

### Fitting a SARIMAX Model with a Change Point     

We can easily include the change point into a SARIMAX model. The code in the cell below creates another exogonous variable that has a binary, $\{0,1\}$ value that changes from $0$ to $1$ past the change point. The extended exogonous set of variables are then used to fit a SARIMAX model using the code below. Notice that the 'predict' method is used here, rather than 'get_forecast' since we are interested in the in-sample predicitons.   

In [ ]:
annual_fourier_terms['shift'] = np.array([0.0]*len(X))
annual_fourier_terms['shift'] = 0.0
annual_fourier_terms.loc[ts_break.index[predicted_change_points.iloc[0].iloc[0]]:, 'shift'] = 1.0


SARIMAX_model_change = SARIMAX(ts_break,
                exog=annual_fourier_terms, # for annual seasonality and change point
                order=(0, 0, 0),
                seasonal_order=(0, 1, 0, 7), # for weekly seasonality 
                enforce_stationarity=False,
                enforce_invertibility=False).fit()

SARIMAX_model_change.summary()

The code in the cell below plots the time series and the SARIMAX prediction for the time series for the 6 months before and 6 month after the change point. 

In [ ]:
predicted = SARIMAX_model_change.predict(exog=annual_fourier_terms['2025-01-01':])    

resid_series = predicted['2023-07-01':'2024-06-30'] - ts_break['2023-07-01':'2024-06-30']
print('RMSE: ' + str(np.std(resid_series)))
print(pd.Series(test_stationarity(resid_series, regression_model='c', rounding=3), 
                   index=['ADF_statistic','ADF_conf_int','ADF_p_value','KPSS_statistic','KPSS_p_value']))

_,ax = plt.subplots(2, 1, figsize=(10,12))
ts_break['2023-07-01':'2024-06-30'].plot(ax=ax[0], c='red', linestyle='--', label='Actual')
predicted['2023-07-01':'2024-06-30'].plot(ax=ax[0])
ax[0].set_title('Time series with break point');
ax[0].set_ylabel('Value');
ax[0].set_xlabel('Date');
ax[0].legend();


pd.Series(resid_series, index=predicted.index)['2023-07-01':'2024-06-30'].plot(ax=ax[1]);
ax[1].set_title('Residual series');
ax[1].set_ylabel('Residual');
ax[1].set_xlabel('Date');

> **Exercise 34-8:** Examine the results of the change point detection example and answer these questions.
> 1. Describe why the break point found corresponds to the statistically significant change in the time series.
> 2. Describe why the break point index detected is in a narrow range.
> 4. Describe at least three types of evidence that the time series model using an exogenous variable to represent before and after the change point.      

> **Answers:**
> 1.    
> 2.    
> 3. Possible answers include:
>    a.    
>    b.      
>    c.     

### Copyright 2023, 2025, 2026 Stephen F Elston. All rights reserved.  